# Step 0: Setup

## Imports

In [ ]:
import pandas as pd
import numpy as np

import ast
import html
import re

### Converting the food.com dataset to DataFrame

In [2]:
df = pd.read_csv('datasets/recipes_ingredients.csv')

In [3]:
df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


### Extracting unique tag values

In [4]:
unique_values = {
    tag.strip(' "') 
    for row in df['tags'] 
    if isinstance(row, str)  
    for tag in row.strip('[]').split(',')
    if tag.strip(' "')      
}
unique_values

{'1-day-or-more',
 '15-minutes-or-less',
 '3-steps-or-less',
 '30-minutes-or-less',
 '4-hours-or-less',
 '5-ingredients-or-less',
 '60-minutes-or-less',
 'Cool Whip',
 'Throw the ultimate fiesta with this sopaipillas recipe from Food.com.',
 'a1-sauce',
 'african',
 'american',
 'amish-mennonite',
 'angolan',
 'appetizers',
 'appetizers-seafood',
 'apple-pie',
 'apples',
 'april-fools-day',
 'argentine',
 'artichoke',
 'asian',
 'asparagus',
 'australian',
 'austrian',
 'avocado',
 'bacon',
 'baja',
 'baked-beans',
 'baking',
 'bananas',
 'bar-cookies',
 'barbecue',
 'bass',
 'bath-beauty',
 'bean-soup',
 'beans',
 'beans-side-dishes',
 'beans-soups',
 'bear',
 'beef',
 'beef-barley-soup',
 'beef-crock-pot',
 'beef-kidney',
 'beef-liver',
 'beef-organ-meats',
 'beef-ribs',
 'beef-sandwiches',
 'beef-sauces',
 'beef-sausage',
 'beginner-cook',
 'beijing',
 'belgian',
 'berries',
 'beverages',
 'birthday',
 'biscotti',
 'bisques-cream-soups',
 'bizarre',
 'black-bean-soup',
 'black-beans

# Step 1: Extracting features from tags

Because the entries are shaped like a Python string, we use `ast.literal_eval` to turn them into actual Python lists, if the value is NaN, make it into an empty list.

In [5]:
def parse_tags(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    return ast.literal_eval(value)

df["tags_parsed"] = df["tags"].apply(parse_tags)

Function that scans for certain tag values and derives feature columns from those, and it does that from a mapping described by a given dictionary `mapping`.
`prefix` is to attach a category specific prefix to the added column, to organize them for later verifications.
`prefix_rules` is a dictionary that makes sure not duplicate columns are added with the wrong prefix

In [6]:
def apply_tag_mapping(
    df,
    mapping,
    parsed_col="tags_parsed",
    prefix=None,
    prefix_rules=None
):
  # Function for creating the new feature column name  
    def final_col_name(feature):
        # if there are prefix rules, 
        # verify if feature starts with a prefix existing in the prefix rules
        # if yes,return the column with the prefix set in `prefix_rules`
        if prefix_rules:
            for starts_with, rule_prefix in prefix_rules.items():
                if feature.startswith(starts_with):
                    return f"{rule_prefix}_{feature}"

        # if there are no prefix rules, just attach the prefix from the `prefix` parameter
        if prefix is not None:
            return f"{prefix}_{feature}"

        return feature


    # Selecting the columns that will be added to the data_frame
    feature_cols = []
    
    for features in mapping.values():
        for f in features:
            col_name = final_col_name(f)
            feature_cols.append(col_name)
    
    feature_cols = sorted(set(feature_cols))
    

    new_feature_cols = []

    for col in feature_cols:
        if col not in df.columns: # verifying if columns exist in dataframe so we don't add duplicates
            new_feature_cols.append(col)

    #adding the new columns
    if new_feature_cols:
        df = pd.concat(
            [df, pd.DataFrame(0, index=df.index, columns=new_feature_cols)],
            axis=1
        )

    # 
    for raw_tag, features in mapping.items():
        # if there is an empty list, don't add anything, this is for ignoring or "removing" irelevant tags
        if not features:
            continue

        mask = df[parsed_col].apply(lambda tags: raw_tag in tags) # a series with true / false values along the rows to mark where the feature is present

        for feature in features:
            df.loc[mask, final_col_name(feature)] = 1 # applying the mask

    return df

## 1.1 Time-based tags

In [7]:
time_tag_mapping = {
    "15-minutes-or-less": ["is_quick_meal"],
    "30-minutes-or-less": ["is_quick_meal"],
    "60-minutes-or-less": ["is_medium_time"],
    "4-hours-or-less": ["is_long_cook"],
    "1-day-or-more": ["is_long_cook"],
    "time-to-make": [],
}

df = apply_tag_mapping(df, time_tag_mapping, prefix="duration")

In [8]:
df.head(5)

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,duration_is_long_cook,duration_is_medium_time,duration_is_quick_meal
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",0,1,0
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",0,1,0
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",1,0,0
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, pre...",0,1,0
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course...","[15-minutes-or-less, time-to-make, course, mai...",0,0,1


## 1.2 Meal Complexity tags

In [9]:
difficulty_tag_mapping = {
    "3-steps-or-less": ["is_easy"],
    "5-ingredients-or-less": ["is_easy"],
    "beginner-cook": ["is_beginner_friendly"],
    "easy": ["is_easy"],
    "from-scratch": ["is_from_scratch"],
    "inexpensive": ["is_budget_friendly"],
    "no-cook": ["is_no_cook"],
    "weeknight": ["is_quick_meal"],
}

df = apply_tag_mapping(df, difficulty_tag_mapping, prefix="complexity")

In [10]:
df.head(5)

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,duration_is_long_cook,duration_is_medium_time,duration_is_quick_meal,complexity_is_beginner_friendly,complexity_is_budget_friendly,complexity_is_easy,complexity_is_from_scratch,complexity_is_no_cook,complexity_is_quick_meal
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",0,1,0,0,0,0,0,0,0
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",0,1,0,0,0,0,0,0,0
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",1,0,0,0,0,1,0,0,0
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, pre...",0,1,0,0,0,0,0,0,0
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course...","[15-minutes-or-less, time-to-make, course, mai...",0,0,1,0,0,0,0,0,0


## 1.3 Meal Type tags

In [11]:
meal_type_mapping = {
    "appetizers": ["is_appetizer"],
    "appetizers-seafood": ["is_appetizer", "has_seafood"],

    "beverages": [],

    "breakfast": ["is_breakfast", "is_complete_meal"],
    "breakfast-casseroles": ["is_breakfast", "is_casserole"],
    "breakfast-eggs": ["is_breakfast", "has_eggs"],
    "breakfast-potatoes": ["is_breakfast", "has_potatoes"],

    "brunch": ["is_brunch"],

    "casseroles": ["is_casserole", "is_complete_meal"],
    "casseroles-one-dish-meal": ["is_casserole", "is_complete_meal"],

    "cocktails": [],
    "condiments-etc": [],
    "course": [],

    "desserts": ["is_dessert"],
    "desserts-easy": ["is_dessert"],
    "desserts-fruit": ["is_dessert", "has_fruit"],

    "dips": [],
    "dips-lunch-snacks": [],
    "dips-summer": [],

    "finger-food": ["is_appetizer"],

    "garnishes": [],

    "lunch": ["is_lunch", "is_complete_meal"],

    "main-dish": ["is_complete_meal"],
    "main-dish-beef": ["is_complete_meal", "has_beef"],
    "main-dish-casseroles": ["is_complete_meal", "is_casserole"],
    "main-dish-chicken": ["is_complete_meal", "has_chicken"],
    "main-dish-crock-pot": ["is_complete_meal", "is_slow_cooker"],
    "main-dish-pasta": ["is_complete_meal", "has_pasta"],
    "main-dish-pork": ["is_complete_meal", "has_pork"],
    "main-dish-seafood": ["is_complete_meal", "has_seafood"],

    "omelets-and-frittatas": [
        "is_breakfast",
        "has_eggs",
        "is_complete_meal"
    ],

    "one-dish-meal": ["is_complete_meal"],
    "pancakes-and-waffles": ["is_breakfast", "is_sweet"],
    "pasta-salad": ["is_salad", "has_pasta"],
    "picnic": [],
    "pizza": ["is_complete_meal", "is_pizza"],
    "potluck": [],
    "salads": ["is_salad"],
    "sandwiches": ["is_sandwich", "is_complete_meal"],
    "side-dishes": [],
    "side-dishes-beans": [],
    "snacks": ["is_snack"],
    "snacks-kid-friendly": ["is_snack"],
    "snacks-sweet": ["is_snack", "is_sweet"],
    "soups-beans": ["is_soup", "has_beans", "is_complete_meal"],
    "soups-crock-pot": ["is_soup", "is_slow_cooker"],
    "soups-stews": ["is_soup", "is_stew"],
    "spreads": [],
}

df = apply_tag_mapping(
    df,
    meal_type_mapping,
    prefix="meal_type",
    prefix_rules={
        "has_": "ingredient"
    }
)

In [12]:
df.head(5)

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,...,meal_type_is_dessert,meal_type_is_lunch,meal_type_is_pizza,meal_type_is_salad,meal_type_is_sandwich,meal_type_is_slow_cooker,meal_type_is_snack,meal_type_is_soup,meal_type_is_stew,meal_type_is_sweet
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,1,0,0,0,0,0,0,0,0,0
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",...,0,0,0,0,0,0,0,0,0,0
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, pre...",...,1,0,0,0,0,0,0,0,0,0
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course...","[15-minutes-or-less, time-to-make, course, mai...",...,1,0,0,0,0,0,0,0,0,0


## 1.4 Cuisine tags

### 1.4.1 Africa

In [13]:
african_cuisine_mapping = {
    "african": ["is_african"],
    "angolan": ["is_african"],
    "congolese": ["is_african"],

    "egyptian": ["is_african", "is_middle_eastern"],

    "ethiopian": ["is_african", "is_ethiopian"],

    "libyan": ["is_african", "is_middle_eastern"],

    "moroccan": ["is_african", "is_moroccan"],

    "namibian": ["is_african"],

    "nigerian": ["is_african", "is_west_african"],

    "somalian": ["is_african", "is_middle_eastern"],

    "south-african": ["is_african", "is_south_african"],

    "sudanese": ["is_african", "is_middle_eastern"],
}

df = apply_tag_mapping(df, african_cuisine_mapping, prefix="cuisine")

### 1.4.2 Asia

In [14]:
asian_cuisine_mapping = {
    "asian": ["is_asian"],

    "beijing": ["is_asian", "is_chinese"],
    "cantonese": ["is_asian", "is_chinese"],
    "chinese": ["is_asian", "is_chinese"],
    "hunan": ["is_asian", "is_chinese"],
    "szechuan": ["is_asian", "is_chinese"],

    "japanese": ["is_asian", "is_japanese"],
    "korean": ["is_asian", "is_korean"],
    "thai": ["is_asian", "is_thai"],
    "vietnamese": ["is_asian", "is_vietnamese"],

    "indian": ["is_asian", "is_indian"],

    "filipino": ["is_asian"],
    "cambodian": ["is_asian"],
    "indonesian": ["is_asian"],
    "malaysian": ["is_asian"],
    "laotian": ["is_asian"],
    "mongolian": ["is_asian"],
    "nepalese": ["is_asian"],

    "pakistani": ["is_asian", "is_indian_subcontinent"],

    "iranian-persian": ["is_middle_eastern"],
    "iraqi": ["is_middle_eastern"],
    "lebanese": ["is_middle_eastern"],
    "palestinian": ["is_middle_eastern"],
    "saudi-arabian": ["is_middle_eastern"],

    "turkish": ["is_middle_eastern", "is_turkish"],

    "middle-eastern": ["is_middle_eastern"],

    "georgian": ["is_european"],
}

df = apply_tag_mapping(df, asian_cuisine_mapping,prefix="cuisine")

### 1.4.3 Europe

In [15]:
european_cuisine_mapping = {
    "austrian": ["is_european"],
    "belgian": ["is_european"],
    "british-columbian": ["is_north_american"],
    "czech": ["is_european"],
    "danish": ["is_european", "is_scandinavian"],
    "dutch": ["is_european"],
    "english": ["is_european", "is_british"],
    "european": ["is_european"],
    "finnish": ["is_european", "is_scandinavian"],
    "french": ["is_european", "is_french"],
    "german": ["is_european", "is_german"],
    "greek": ["is_european", "is_mediterranean", "is_greek"],
    "hungarian": ["is_european"],
    "icelandic": ["is_european", "is_scandinavian"],
    "irish": ["is_european", "is_british"],
    "italian": ["is_european", "is_mediterranean", "is_italian"],
    "norwegian": ["is_european", "is_scandinavian"],
    "polish": ["is_european"],
    "portuguese": ["is_european", "is_mediterranean"],
    "russian": ["is_european", "is_russian"],
    "scandinavian": ["is_european", "is_scandinavian"],
    "scottish": ["is_european", "is_british"],
    "spanish": ["is_european", "is_mediterranean", "is_spanish"],
    "swedish": ["is_european", "is_scandinavian"],
    "swiss": ["is_european"],
    "welsh": ["is_european", "is_british"],
}

df = apply_tag_mapping(df, european_cuisine_mapping,prefix="cuisine")

### 1.4.5 Americas

In [16]:
american_cuisine_mapping = {
    "american": ["is_american"],
    "amish-mennonite": ["is_american"],
    "baja": ["is_mexican"],
    "californian": ["is_american"],
    "canadian": ["is_north_american"],
    "cajun": ["is_american"],
    "central-american": ["is_central_american"],
    "colombian": ["is_south_american"],
    "costa-rican": ["is_central_american"],
    "cuban": ["is_caribbean"],
    "guatemalan": ["is_central_american"],
    "hawaiian": ["is_american"],
    "honduran": ["is_central_american"],
    "mexican": ["is_mexican"],
    "native-american": ["is_american"],
    "north-american": ["is_north_american"],
    "oaxacan": ["is_mexican"],
    "pacific-northwest": ["is_american"],
    "pennsylvania-dutch": ["is_american"],
    "puerto-rican": ["is_caribbean"],
    "soul": ["is_american"],
    "southern-united-states": ["is_american"],
    "southwestern-united-states": ["is_american", "is_mexican"],
    "tex-mex": ["is_american", "is_mexican"],
    "argentine": ["is_south_american"],
    "brazilian": ["is_south_american", "is_brazilian"],
    "caribbean": ["is_caribbean"],
    "chilean": ["is_south_american"],
    "ecuadorean": ["is_south_american"],
    "peruvian": ["is_south_american", "is_peruvian"],
    "south-american": ["is_south_american"],
    "venezuelan": ["is_south_american"],
}

df = apply_tag_mapping(df, american_cuisine_mapping,prefix="cuisine")

### 1.4.6 Oceania / Pacific

In [17]:
oceania_cuisine_mapping = {
    "australian": ["is_oceanian"],
    "micro-melanesia": ["is_oceanian"],
    "new-zealand": ["is_oceanian"],
    "polynesian": ["is_oceanian"],
    "south-west-pacific": ["is_oceanian"],
}

df = apply_tag_mapping(df, oceania_cuisine_mapping,prefix="cuisine")

In [18]:
df.head(3)

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,...,cuisine_is_scandinavian,cuisine_is_spanish,cuisine_is_american,cuisine_is_brazilian,cuisine_is_caribbean,cuisine_is_central_american,cuisine_is_mexican,cuisine_is_peruvian,cuisine_is_south_american,cuisine_is_oceanian
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",...,0,0,1,0,0,0,0,0,0,0


## 1.5 Diet

In [19]:
diet_nutrition_mapping = {
    "dairy-free": ["is_dairy_free"],
    "diabetic": [],

    "egg-free": ["is_egg_free"],
    "free-of-something": [],

    "gluten-free": ["is_gluten_free"],
    "gluten-free-appetizers": ["is_gluten_free"],

    "healthy": ["is_healthy"],
    "healthy-2": ["is_healthy"],

    "high-calcium": ["is_high_calcium"],
    "high-fiber": ["is_high_fiber"],
    "high-in-something": [],
    "high-in-something-diabetic-friendly": [],
    "high-protein": ["is_high_protein"],

    "kosher": [],
    "lactose": [],

    "low-calorie": ["is_low_calorie"],
    "low-carb": ["is_low_carb"],
    "low-cholesterol": ["is_low_cholesterol"],
    "low-fat": ["is_low_fat"],
    "low-in-something": [],
    "low-protein": ["is_low_protein"],
    "low-saturated-fat": ["is_low_saturated_fat"],
    "low-sodium": ["is_low_sodium"],

    "no-shell-fish": [],
    "nut-free": ["is_nut_free"],

    "vegan": ["is_vegan"],
    "vegetarian": ["is_vegetarian"],

    "very-low-carbs": ["is_low_carb"],
}

df = apply_tag_mapping(
    df,
    diet_nutrition_mapping,
    prefix="diet"
)

## 1.6 Ingredient tags

### 1.6.1 Meat

In [20]:
meat_poultry_mapping = {
    "bacon": ["has_pork"],
    "bear": [],

    "beef": ["has_beef"],
    "beef-kidney": ["has_beef"],
    "beef-liver": ["has_beef"],
    "beef-organ-meats": ["has_beef"],
    "beef-ribs": ["has_beef"],
    "beef-sausage": ["has_beef"],
    "brisket": ["has_beef"],
    "ground-beef": ["has_beef"],
    "roast-beef": ["has_beef"],
    "steak": ["has_beef"],
    "steaks": ["has_beef"],
    "veal": ["has_beef"],

    "chicken": ["has_chicken"],
    "chicken-breasts": ["has_chicken"],
    "chicken-livers": ["has_chicken"],
    "chicken-thighs-legs": ["has_chicken"],
    "whole-chicken": ["has_chicken"],
    "wings": ["has_chicken"],

    "duck": ["has_duck"],
    "duck-breasts": ["has_duck"],
    "whole-duck": ["has_duck"],

    "ham": ["has_pork"],
    "hot-dogs": ["has_pork"],
    "pork": ["has_pork"],
    "pork-chops": ["has_pork"],
    "pork-loin": ["has_pork"],
    "pork-loins": ["has_pork"],
    "pork-ribs": ["has_pork"],
    "pork-sausage": ["has_pork"],

    "turkey": ["has_turkey"],
    "turkey-breasts": ["has_turkey"],
    "whole-turkey": ["has_turkey"],

    "deer": [],
    "elk": [],
    "goose": [],
    "lamb-sheep": [],
    "meat": [],
    "moose": [],
    "pheasant": [],
    "poultry": [],
    "quail": [],
    "rabbit": [],
    "wild-game": [],
}

df = apply_tag_mapping(
    df,
    meat_poultry_mapping,
    prefix="ingredient"
)

### 1.6.2 Fish & Seafood

In [21]:
seafood_mapping = {
    "bass": ["has_fish"],
    "catfish": ["has_fish"],
    "clams": ["has_shellfish"],
    "cod": ["has_fish"],

    "crab": ["has_seafood", "has_crab"],
    "crawfish": ["has_shellfish"],

    "fish": ["has_fish"],
    "fish-halibut": ["has_fish"],
    "fish-salmon": ["has_fish", "has_salmon"],
    "fish-tuna": ["has_fish", "has_tuna"],
    "freshwater-fish": ["has_fish"],

    "halibut": ["has_fish"],
    "lobster": ["has_shellfish"],
    "mahi-mahi": ["has_fish"],
    "mussels": ["has_shellfish"],

    "octopus": ["has_seafood", "has_octopus"],

    "orange-roughy": ["has_fish"],
    "oysters": ["has_shellfish"],
    "perch": ["has_fish"],
    "pickeral": ["has_fish"],

    "salmon": ["has_fish", "has_salmon"],
    "saltwater-fish": ["has_fish"],
    "scallops": ["has_shellfish"],

    "seafood": ["has_seafood"],
    "shellfish": ["has_seafood", "has_shellfish"],
    "shrimp": ["has_seafood", "has_shrimp"],

    "sole-and-flounder": ["has_fish"],
    "squid": ["has_seafood"],
    "tilapia": ["has_fish"],
    "trout": ["has_fish"],
    "tuna": ["has_fish", "has_tuna"],
    "whitefish": ["has_fish"],
}

df = apply_tag_mapping(
    df,
    seafood_mapping,
    prefix="ingredient"
)

### 1.6.3 Vegetables

In [22]:
vegetable_mapping = {
    "artichoke": ["has_vegetables"],

    "asparagus": ["has_vegetables"],

    "avocado": ["has_avocado", "has_vegetables"],

    "beans": ["has_legumes", "has_vegetables"],
    "black-beans": ["has_legumes", "has_vegetables"],
    "green-yellow-beans": ["has_legumes", "has_vegetables"],
    "lentils": ["has_legumes", "has_vegetables"],
    "chick-peas-garbanzos": ["has_legumes", "has_vegetables"],

    "bok-choys": ["has_leafy_greens", "has_vegetables"],
    "chard": ["has_leafy_greens", "has_vegetables"],
    "collard-greens": ["has_leafy_greens", "has_vegetables"],
    "greens": ["has_leafy_greens", "has_vegetables"],
    "lettuces": ["has_leafy_greens", "has_vegetables"],
    "spinach": ["has_leafy_greens", "has_vegetables"],

    "broccoli": ["has_cruciferous_vegetables", "has_vegetables"],
    "cabbage": ["has_cruciferous_vegetables", "has_vegetables"],
    "cauliflower": ["has_cruciferous_vegetables", "has_vegetables"],

    "carrots": ["has_carrots", "has_vegetables"],

    "corn": ["has_vegetables"],

    "eggplant": ["has_vegetables"],

    "mushrooms": ["has_mushrooms", "has_vegetables"],

    "onions": ["has_onions", "has_vegetables"],

    "peppers": ["has_peppers", "has_vegetables"],

    "potatoes": ["has_potatoes", "has_vegetables"],
    "yams-sweet-potatoes": ["has_potatoes", "has_vegetables"],

    "pumpkin": ["has_vegetables"],

    "soy-tofu": ["has_tofu", "has_vegetables"],

    "squash": ["has_vegetables"],

    "tomatoes": ["has_tomatoes", "has_vegetables"],

    "vegetables": ["has_vegetables"],

    "zucchini": ["has_zucchini", "has_vegetables"],
}

df = apply_tag_mapping(
    df,
    vegetable_mapping,
    prefix="ingredient"
)

### 1.6.4 Fruits

In [23]:
fruit_mapping = {
    "apples": ["has_fruit", "has_apples"],
    "bananas": ["has_fruit", "has_bananas"],

    "berries": ["has_fruit", "has_berries"],
    "blueberries": ["has_fruit", "has_berries"],
    "cranberries": ["has_fruit", "has_berries"],
    "raspberries": ["has_fruit", "has_berries"],
    "strawberries": ["has_fruit", "has_berries"],

    "cherries": ["has_fruit"],
    "grapes": ["has_fruit"],
    "kiwifruit": ["has_fruit"],
    "melons": ["has_fruit"],
    "peaches": ["has_fruit"],
    "pears": ["has_fruit"],
    "pitted-fruit": ["has_fruit"],
    "plums": ["has_fruit"],

    "citrus": ["has_fruit", "has_citrus"],
    "lemon": ["has_fruit", "has_citrus"],
    "lime": ["has_fruit", "has_citrus"],
    "limes": ["has_fruit", "has_citrus"],
    "oranges": ["has_fruit", "has_citrus"],

    "coconut": ["has_fruit", "has_coconut"],

    "mango": ["has_fruit", "has_tropical_fruit"],
    "papaya": ["has_fruit", "has_tropical_fruit"],
    "pineapple": ["has_fruit", "has_tropical_fruit"],
    "tropical-fruit": ["has_fruit", "has_tropical_fruit"],

    "peanut-butter": [],

    "pumpkin": ["has_vegetables"],
}

df = apply_tag_mapping(
    df,
    fruit_mapping,
    prefix="ingredient"
)

### 1.6.5 Grains

In [24]:
grains_mapping = {
    "breads": ["has_bread", "has_grains"],

    "brown-rice": ["has_rice", "has_grains"],
    "long-grain-rice": ["has_rice", "has_grains"],
    "medium-grain-rice": ["has_rice", "has_grains"],
    "rice": ["has_rice", "has_grains"],
    "short-grain-rice": ["has_rice", "has_grains"],
    "white-rice": ["has_rice", "has_grains"],

    "elbow-macaroni": ["has_pasta", "has_grains"],
    "lasagna": ["has_pasta", "has_grains"],
    "lasagne": ["has_pasta", "has_grains"],
    "macaroni-and-cheese": ["has_pasta", "has_grains"],
    "manicotti": ["has_pasta", "has_grains"],
    "pasta": ["has_pasta", "has_grains"],
    "pasta-elbow-macaroni": ["has_pasta", "has_grains"],
    "pasta-rice-and-grains-elbow-macaroni": ["has_pasta", "has_grains"],
    "pasta-shells": ["has_pasta", "has_grains"],
    "penne": ["has_pasta", "has_grains"],
    "ravioli-tortellini": ["has_pasta", "has_grains"],
    "spaghetti": ["has_pasta", "has_grains"],

    "grains": ["has_grains"],
    "pasta-rice-and-grains": ["has_grains"],

    "granola-and-porridge": ["has_grains", "has_oatmeal"],
    "oatmeal": ["has_oatmeal", "has_grains"],

    "quick-breads": ["has_bread", "has_grains"],
    "sourdough": ["has_bread", "has_grains"],
    "wheat-bread": ["has_bread", "has_grains"],

    "yeast": [],
}

df = apply_tag_mapping(
    df,
    grains_mapping,
    prefix="ingredient"
)

### 1.6.6 Dairy

In [25]:
dairy_eggs_mapping = {
    "butter": ["has_dairy"],

    "cheese": ["has_dairy", "has_cheese"],

    "cream": ["has_dairy"],

    "egg whites": ["has_eggs"],
    "eggs": ["has_eggs"],
    "eggs-dairy": ["has_eggs", "has_dairy"],

    "ice-cream": ["has_dairy"],

    "lime yogurt": ["has_dairy"],

    "margarine": ["has_fats"],
}

df = apply_tag_mapping(
    df,
    dairy_eggs_mapping,
    prefix="ingredient",
    prefix_rules={
        "is_": "meal_type"
    }
)

## 1.7 Replacing sauces with tastes

In [26]:
flavor_pantry_mapping = {
    "a1-sauce": ["is_savory"],

    "brown sugar": ["is_sweet"],
    "sugar": ["is_sweet"],
    "sweet-sauces": ["is_sweet"],

    "canned chipotle chiles": ["is_spicy"],
    "chili powder": ["is_spicy"],
    "spices": ["is_spicy"],

    "garlic cloves": ["is_savory"],
    "herb-and-spice-mixes": ["is_savory"],
    "marinara-sauce": ["is_savory"],
    "marinades-and-rubs": ["is_savory"],
    "savory-sauces": ["is_savory"],
    "tomato-sauce": ["is_savory"],
    "tomatoes-sauces": ["is_savory"],

    "juice and zest of": ["has_citrus"],
    "lime zest": ["has_citrus"],

    "kosher salt": ["is_salty"],

    "vegetable oil": ["has_fats"],

    "sauces": [],
    "water": [],
}

df = apply_tag_mapping(
    df,
    flavor_pantry_mapping,
    prefix="taste",
    prefix_rules={
        "has_": "ingredient"
    }
)

## 1.8 Cooking method tags

In [27]:
cooking_method_mapping = {
    "baking": ["is_baked"],

    "barbecue": ["is_grilled"],
    "broil": ["is_grilled"],
    "grilling": ["is_grilled"],

    "canning": [],
    "pressure-canning": [],

    "deep-fry": ["is_fried"],

    "microwave": ["is_microwaved"],

    "steam": ["is_steamed"],

    "stir-fry": ["is_stir_fry"],

    "stove-top": [],
}

df = apply_tag_mapping(
    df,
    cooking_method_mapping,
    prefix="cooking"
)

## 1.9 Meal Equipment

In [28]:
equipment_mapping = {
    "bread-machine": ["is_baked"],

    "crock-pot-slow-cooker": ["is_slow_cooked"],
    "slow-cooker": ["is_slow_cooked"],

    "dehydrator": [],

    "food-processor-blender": ["is_blended"],

    "freezer": [],
    "mixer": [],
    "refrigerator": [],
    "small-appliance": [],

    "oven": ["is_baked"],

    "pressure-cooker": ["is_pressure_cooked"],

    "smoker": ["is_smoked"],
}

df = apply_tag_mapping(
    df,
    equipment_mapping,
    prefix="cooking"
)

## 1.10 Dish type

In [29]:
dish_type_mapping = {
    "bar-cookies": ["is_dessert"],
    "biscotti": ["is_dessert"],
    "brownies": ["is_dessert"],
    "cakes": ["is_dessert"],
    "candy": ["is_dessert"],
    "cheesecake": ["is_dessert"],
    "chocolate-chip-cookies": ["is_dessert"],
    "cobblers-and-crisps": ["is_dessert"],
    "coffee-cakes": ["is_dessert"],
    "cookie": ["is_dessert"],
    "cookies-and-brownies": ["is_dessert"],
    "cupcakes": ["is_dessert"],
    "fudge": ["is_dessert"],
    "key-lime-pie": ["is_dessert"],
    "lemon-cake": ["is_dessert"],
    "muffins": ["is_dessert"],
    "no-bake-cookies": ["is_dessert"],
    "peanut-butter-pie": ["is_dessert"],
    "pies": ["is_dessert"],
    "pies-and-tarts": ["is_dessert"],
    "puddings-and-mousses": ["is_dessert"],
    "scones": ["is_dessert"],
    "sugar-cookies": ["is_dessert"],
    "tarts": ["is_dessert"],

    "bean-soup": ["is_soup"],
    "beef-barley-soup": ["is_soup"],
    "bisques-cream-soups": ["is_soup"],
    "black-bean-soup": ["is_soup"],
    "chowders": ["is_soup"],
    "clear-soups": ["is_soup"],
    "mushroom-soup": ["is_soup"],
    "navy-bean-soup": ["is_soup"],
    "potato-soup": ["is_soup"],
    "tortilla-soup": ["is_soup"],

    "chili": ["is_stew"],
    "gumbo": ["is_stew"],
    "stews": ["is_stew"],

    "beef-sandwiches": ["is_sandwich"],
    "burgers": ["is_burger", "is_complete_meal"],

    "crab": ["has_seafood", "has_crab"],

    "curries": ["is_curry", "is_complete_meal"],
    "curries-indian": ["is_curry"],

    "macaroni-and-cheese": ["has_pasta", "is_complete_meal"],
    "pasta-salad": ["is_salad", "has_pasta"],

    "meatballs": ["is_complete_meal"],
    "meatloaf": ["is_complete_meal"],
    "pizza": ["is_pizza", "is_complete_meal"],
    "pot-pie": ["is_complete_meal"],
    "pot-roast": ["is_complete_meal"],
    "quiche": ["is_complete_meal"],

    "pretzels": ["is_snack"],
    "rolls-biscuits": ["has_bread"],

    "shakes": ["is_beverage"],
    "smoothies": ["is_beverage", "is_light_meal"],

    "beef-sauces": [],
    "jams-and-preserves": [],
    "jellies": [],
    "margarita": [],
    "mashed-potatoes": [],
    "punch": [],
    "salsas": [],
    "spaghetti-sauce": [],
    "stocks": [],
}

df = apply_tag_mapping(
    df,
    dish_type_mapping,
    prefix="meal_type",
    prefix_rules={
        "has_": "ingredient",
        "is_indian": "cuisine"
    }
)

In [30]:
df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,...,cooking_is_steamed,cooking_is_stir_fry,cooking_is_blended,cooking_is_pressure_cooked,cooking_is_slow_cooked,cooking_is_smoked,meal_type_is_beverage,meal_type_is_burger,meal_type_is_curry,meal_type_is_light_meal
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",...,0,0,0,0,0,0,0,0,0,0
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, pre...",...,0,0,0,0,0,0,0,0,0,0
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course...","[15-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0


## 1.11 Removing holidays

In [31]:
holiday_mapping = {
    "april-fools-day": [],
    "christmas": [],
    "chinese-new-year": [],
    "cinco-de-mayo": [],
    "easter": [],
    "fathers-day": [],
    "fourth-of-july": [],
    "halloween": [],
    "hanukkah": [],
    "independence-day": [],
    "kwanzaa": [],
    "labor-day": [],
    "mardi-gras-carnival": [],
    "memorial-day": [],
    "mothers-day": [],
    "new-years": [],
    "passover": [],
    "ramadan": [],
    "rosh-hashana": [],
    "rosh-hashanah": [],
    "st-patricks-day": [],
    "super-bowl": [],
    "superbowl": [],
    "thanksgiving": [],
    "valentines-day": [],
    "wedding": [],
}

df = apply_tag_mapping(df, holiday_mapping)

## 1.12 Seasons

In [32]:
season_mapping = {
    "fall": ["is_fall"],
    "seasonal": [],
    "spring": ["is_spring"],
    "summer": ["is_summer"],
    "winter": ["is_winter"],
}

df = apply_tag_mapping(
    df,
    season_mapping,
    prefix="season"
)

## 1.13 Removing Lifestyle Context

In [33]:
lifestyle_context_mapping = {
    "birthday": [],
    "camping": [],
    "college": [],
    "comfort-food": [],
    "dinner-party": [],
    "for-large-groups": [],
    "gifts": [],
    "holiday-event": [],
    "kid-friendly": [],
    "leftovers": [],
    "picnic": [],
    "romantic": [],
    "toddler-friendly": [],
    "to-go": [],
}

df = apply_tag_mapping(df, lifestyle_context_mapping)

## 1.14 Serving style tags

In [34]:
serving_style_mapping = {
    "brown-bag": [],

    "finger-food": ["is_appetizer"],

    "flat-shapes": [],
    "presentation": [],

    "served-cold": ["is_served_cold"],
    "served-hot": ["is_served_hot"],
}

df = apply_tag_mapping(
    df,
    serving_style_mapping,
    prefix="temp",
    prefix_rules={
        "is_appetizer": "meal_type"
    }
)

## 1.15 Removing branded items

In [35]:
branded_noise_mapping = {
    "Cool Whip": [],
    "hidden-valley-ranch": [],
    "queso-for-all": [],
    "ragu-recipe-contest": [],
    "reynolds-wrap": [],
    "simply-potatoes": [],
    "simply-potatoes2": [],
}

df = apply_tag_mapping(df, branded_noise_mapping)

## 1.16 Removing noisy metadata

In [36]:
meta_system_noise_mapping = {
    "bath-beauty": [],
    "bizarre": [],
    "celebrity": [],
    "cuisine": [],
    "dietary": [],
    "equipment": [],
    "heirloom-historical": [],
    "heirloom-historical-recipes": [],
    "homeopathy-remedies": [],
    "main-ingredient": [],
    "non-food-products": [],
    "novelty": [],
    "number-of-servings": [],
    "occasion": [],
    "preparation": [],
    "recipes": [],
    "recipies": [],
    "taste-mood": [],
    "technique": [],
}

df = apply_tag_mapping(df, meta_system_noise_mapping)

## 1.17. Removing corrupted noise

In [37]:
corrupted_noise_mapping = {
    "Throw the ultimate fiesta with this sopaipillas recipe from Food.com.": [],
    "less_thansql:name_topics_of_recipegreater_than": [],
    "green food coloring recipe": [],
    "vegetable oil recipe": [],
    "white cake mix": [],
    "canned chipotle chiles": [],
    "juice and zest of": [],
}

df = apply_tag_mapping(df, corrupted_noise_mapping)

# Step 2: Further analysis

## 2.1 Checking total number of columns

In [38]:
len(list(df.columns))

152

## 2.2 Keeping only meals (removing drinks, sauces, etc.)

In [39]:
candidate_df = df[
    (
        (df["meal_type_is_complete_meal"] == 1)
        |
        (df["meal_type_is_salad"] == 1)
        |
        (df["meal_type_is_soup"] == 1)
        |
        (df["meal_type_is_light_meal"] == 1)
    )
]

## 2.2 Checking recipe frequency for each feature

In [40]:
feature_cols = [col for col in candidate_df.columns if col.startswith((
    "ingredient_",
    "meal_type_",
    "cuisine_",
    "diet_",
    "cooking_",
    "duration_",
    "complexity_",
    "taste_",
    "season_",
    "taste_"
))]

pd.set_option("display.max_rows", None)


candidate_df[feature_cols].sum().sort_values()

taste_is_spicy                                0
taste_is_salty                                0
ingredient_has_fats                           0
ingredient_has_beans                          1
meal_type_is_slow_cooker                      2
ingredient_has_octopus                       27
cuisine_is_west_african                      47
cuisine_is_ethiopian                         60
diet_is_high_fiber                           76
cuisine_is_peruvian                         139
diet_is_dairy_free                          143
taste_is_sweet                              144
cuisine_is_indian_subcontinent              147
cuisine_is_brazilian                        194
cooking_is_smoked                           214
ingredient_has_avocado                      232
cuisine_is_south_african                    233
cuisine_is_turkish                          248
ingredient_has_zucchini                     289
cuisine_is_korean                           353
ingredient_has_duck                     

In [41]:
rare_features = candidate_df[feature_cols].sum().sort_values()

rare_features[rare_features < 100]

taste_is_spicy               0
taste_is_salty               0
ingredient_has_fats          0
ingredient_has_beans         1
meal_type_is_slow_cooker     2
ingredient_has_octopus      27
cuisine_is_west_african     47
cuisine_is_ethiopian        60
diet_is_high_fiber          76
dtype: int64

## 2.3 Taste Inference

### 2.3.1 Sweet

In [42]:
candidate_df.loc[
    (
        (candidate_df["meal_type_is_dessert"] == 1) |
        (candidate_df["ingredient_has_fruit"] == 1)
    ),
    "taste_is_sweet"
] = 1

### 2.3.2 Savory

In [43]:
savory_features = [
    # proteins
    "ingredient_has_beef",
    "ingredient_has_chicken",
    "ingredient_has_pork",
    "ingredient_has_turkey",
    "ingredient_has_duck",
    "ingredient_has_fish",
    "ingredient_has_seafood",
    "ingredient_has_shellfish",
    "ingredient_has_crab",
    "ingredient_has_shrimp",
    "ingredient_has_salmon",
    "ingredient_has_tuna",
    "ingredient_has_eggs",
    "ingredient_has_tofu",
    "ingredient_has_legumes",

    # savory vegetables / aromatics
    "ingredient_has_mushrooms",
    "ingredient_has_onions",
    "ingredient_has_peppers",
    "ingredient_has_tomatoes",
    "ingredient_has_potatoes",

    # dairy / grains often used in savory dishes
    "ingredient_has_cheese",
    "ingredient_has_dairy",
    "ingredient_has_pasta",
    "ingredient_has_rice",
    "ingredient_has_bread",

    # meal types
    "meal_type_is_complete_meal",
    "meal_type_is_soup",
    "meal_type_is_stew",
    "meal_type_is_casserole",
    "meal_type_is_curry",
    "meal_type_is_burger",
    "meal_type_is_sandwich",
    "meal_type_is_pizza",
    "meal_type_is_salad",

    # cooking methods
    "cooking_is_grilled",
    "cooking_is_fried",
    "cooking_is_baked",
    "cooking_is_stir_fry",
    "cooking_is_slow_cooked",
    "cooking_is_smoked",
]

savory_features = [col for col in savory_features if col in candidate_df.columns]

In [44]:
candidate_df["taste_is_savory"] = (
    candidate_df[savory_features].sum(axis=1) > 0
).astype(int)

# to avoid recipes that have bread/dairy/are baked but are desserts
candidate_df.loc[
    candidate_df["meal_type_is_dessert"] == 1,
    "taste_is_savory"
] = 0

### 2.3.3 Spicy

In [45]:
candidate_df.loc[
    (
        (candidate_df["cuisine_is_mexican"] == 1) |
        (candidate_df["cuisine_is_indian"] == 1) |
        (candidate_df["meal_type_is_curry"] == 1)
    ),
    "taste_is_spicy"
] = 1

## 2.3.4 Checking taste distribution

In [46]:
taste_cols = [
    col for col in candidate_df.columns
    if col.startswith("taste_")
]

taste_distribution = (
    candidate_df[taste_cols]
    .sum()
    .sort_values(ascending=False)
)

print(taste_distribution)

taste_is_savory    247578
taste_is_sweet      21632
taste_is_spicy      18056
taste_is_salty          0
dtype: int64


In [47]:
taste_percentages = (
    candidate_df[taste_cols]
    .mean() * 100
).sort_values(ascending=False)

print(taste_percentages)

taste_is_savory    95.354704
taste_is_sweet      8.331568
taste_is_spicy      6.954271
taste_is_salty      0.000000
dtype: float64


## 2.4 Removing irelevant columns

### Selecting features with under 100 recipes

In [48]:
feature_counts = candidate_df[feature_cols].sum()

rare_features = feature_counts[
    feature_counts < 100
].index.tolist()

rare_features

['ingredient_has_beans',
 'meal_type_is_slow_cooker',
 'cuisine_is_ethiopian',
 'cuisine_is_west_african',
 'diet_is_high_fiber',
 'ingredient_has_octopus',
 'ingredient_has_fats',
 'taste_is_salty']

### Removing them from the dataset 

In [49]:
candidate_df = candidate_df.drop(
    columns=rare_features
)

feature_cols = [col for col in feature_cols if col not in rare_features ]

In [50]:
pd.set_option("display.max_rows", 20)
candidate_df[feature_cols].sum().sort_values()

cuisine_is_peruvian                  139
diet_is_dairy_free                   143
cuisine_is_indian_subcontinent       147
cuisine_is_brazilian                 194
cooking_is_smoked                    214
                                   ...  
duration_is_medium_time            82929
duration_is_quick_meal            101331
complexity_is_easy                137356
meal_type_is_complete_meal        210907
taste_is_savory                   247578
Length: 132, dtype: int64

### Checking number of rows in dataset

In [51]:
len(candidate_df)

259639

### Additional artifact cleanup 

In [52]:
candidate_df[
    candidate_df["name"].str.len() < 5
][["name", "tags"]].head(50)

,name,tags
20877,Jook,"[""time-to-make"", ""course"", ""main-ingredient"", ..."
25107,---,"[""15-minutes-or-less"", ""time-to-make"", ""course..."
29123,More,"[""60-minutes-or-less"", ""time-to-make"", ""course..."
29329,Goop,"[""30-minutes-or-less"", ""time-to-make"", ""course..."
39043,Dahl,"[""curries"", ""60-minutes-or-less"", ""time-to-mak..."
...,...,...
409566,Soup,"[""30-minutes-or-less"", ""time-to-make"", ""course..."
413707,Dal,"[""time-to-make"", ""course"", ""main-ingredient"", ..."
422465,Kima,"[""60-minutes-or-less"", ""time-to-make"", ""course..."
426150,Larb,"[""30-minutes-or-less"", ""time-to-make"", ""course..."


In [53]:
candidate_df = candidate_df[
    ~candidate_df["name"].str.fullmatch(
        r"[-. ]+",
        na=False
    )
]

In [54]:
def clean_recipe_name(name):
    if not isinstance(name, str):
        return name

    name = html.unescape(name)

    # repeated punctuation artifacts
    name = re.sub(r"\.{2,}", " ", name)

    # normalizing quotes/dashes a bit
    name = name.replace("–", "-").replace("—", "-")

    # removing extra whitespace
    name = re.sub(r"\s+", " ", name).strip()

    return name

candidate_df["name"] = candidate_df["name"].apply(clean_recipe_name)

In [55]:
candidate_df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,tags_parsed,...,meal_type_is_beverage,meal_type_is_burger,meal_type_is_curry,meal_type_is_light_meal,season_is_fall,season_is_spring,season_is_summer,season_is_winter,temp_is_served_cold,temp_is_served_hot
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course...","[60-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
6,489452,Teriyaki Pork Chops,I made these on a whim and they are my husband...,"[""teriyaki sauce"", ""pork chops""]","[""1 (16 ounce) bottle teriyaki sauce"",""4 ...","[""I like to marinade them overnight in a ziplo...",4.0,1 (313 g),"[""weeknight"", ""15-minutes-or-less"", ""time-to-m...","[weeknight, 15-minutes-or-less, time-to-make, ...",...,0,0,0,0,0,0,0,0,0,0
8,306467,Quick Bolognese Sauce,Quick Bolognese Sauce Recipe from Festival Foo...,"[""olive oil"", ""yellow onion"", ""celery rib"", ""c...","[""2 tablespoons light olive oil"",""1 sma...","[""Saute onion, celery, and carrot in olive oil...",6.0,1 (297 g),"[""time-to-make"", ""course"", ""main-ingredient"", ...","[time-to-make, course, main-ingredient, cuisin...",...,0,0,0,0,0,0,0,0,0,0
10,457658,Mom's Macaroni Salad,My mom's macaroni salad recipe. I leave out t...,"[""macaroni"", ""mayonnaise"", ""vinegar"", ""yellow ...","[""8 ounces elbow macaroni (cooked)"",""1 ...","[""Boil macaroni according to package direction...",4.0,1 (120 g),"[""30-minutes-or-less"", ""time-to-make"", ""course...","[30-minutes-or-less, time-to-make, course, mai...",...,0,0,0,0,0,0,0,0,0,0
11,437096,Junior League- Fiesta Chicken Salad W/ Lime Ci...,Fiesta Chicken Salad and Lime Cilantro Vinaigr...,"[""chopped shallot"", ""fresh lime juice"", ""chopp...","[""1/2 cup chopped shallot"",""1/4 cup fre...","[""For the vinaigrette, combine the shallots, l...",6.0,1 (212 g),"[""weeknight"", ""30-minutes-or-less"", ""time-to-m...","[weeknight, 30-minutes-or-less, time-to-make, ...",...,0,0,0,0,0,0,0,0,0,0


In [56]:
candidate_df.to_csv("datasets/v1_recipe_dataset.csv")

In [72]:
len(candidate_df)

259627

# Testing users

In [57]:
def recommend_for_user(candidate_df, user_profile, k=10):

    filtered_df = candidate_df.copy()

    # Apply hard constraints
    for feature, required_value in user_profile["hard_filters"].items():

        if feature in filtered_df.columns:

            filtered_df = filtered_df[
                filtered_df[feature] == required_value
            ]

    # Compute weighted preference score
    score = 0

    for feature, weight in user_profile["preferences"].items():

        if feature in filtered_df.columns:

            score += filtered_df[feature] * weight

    filtered_df = filtered_df.copy()

    filtered_df["score"] = score

    recommendations = filtered_df.sort_values(
        "score",
        ascending=False
    ).head(k)

    return recommendations


def show_top_k_reccs(k):
    recommendations = candidate_df.sort_values(
    "score",
    ascending=False
).head(k)

    # for i, recipe in recommendations.iterrows():
    #     print(f"{recipe['id']}: {recipe['name'].title()}")

    return recommendations

In [58]:
def prettify_feature(feature):
    feature = feature.replace("ingredient_has_", "contains ")
    feature = feature.replace("meal_type_is_", "")
    feature = feature.replace("cuisine_is_", "")
    feature = feature.replace("diet_is_", "")
    feature = feature.replace("cooking_is_", "")
    feature = feature.replace("duration_is_", "")
    feature = feature.replace("complexity_is_", "")
    feature = feature.replace("taste_is_", "")

    feature = feature.replace("_", " ")

    return feature.capitalize()


def explain_recommendation(recipe_row, preferences, max_reasons=5):
    matched_preferences = []
    matched_avoidances = []

    for feature, weight in preferences.items():
        if feature not in recipe_row.index:
            continue

        if recipe_row[feature] != 1:
            continue

        readable_feature = prettify_feature(feature)

        if weight > 0:
            matched_preferences.append((readable_feature, weight))
        elif weight < 0:
            matched_avoidances.append((readable_feature, weight))

    matched_preferences = sorted(matched_preferences, key=lambda x: x[1], reverse=True)
    matched_avoidances = sorted(matched_avoidances, key=lambda x: abs(x[1]), reverse=True)

    explanation = []

    for feature, weight in matched_preferences[:max_reasons]:
        explanation.append(f"Matched preference: {feature} (+{weight})")

    for feature, weight in matched_avoidances[:max_reasons]:
        explanation.append(f"Contains avoided feature: {feature} ({weight})")

    return explanation

In [73]:
def show_reccommendations(user):
    recommendations = recommend_for_user(candidate_df, user, k=20)
    
    for index, row in recommendations.iterrows():
        print(row["name"])
        print(explain_recommendation(row, user["preferences"]))
        print("------------------------------------------------------------")

## User #1: Gym Bro

Person representing **Gym Bro** stereotype. A male in his late 20s - early 30s, has as main goal to put on muscle mass. Loves protein-heavy bowls, grilled meats, spicy food, meal prep style eating.

In [60]:
gym_user = {
    "hard_filters": {},

    "preferences": {
        "diet_is_high_protein": 3,
        "ingredient_has_chicken": 2,
        "ingredient_has_beef": 2,
        "cuisine_is_mexican": 4,
        "meal_type_is_complete_meal": 2,
        "meal_type_is_dessert": -5,
    }
}

In [74]:
show_reccommendations(gym_user)

Chicken Enchilada Filling
['Matched preference: Mexican (+4)', 'Matched preference: High protein (+3)', 'Matched preference: Contains chicken (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Who By Fire?
['Matched preference: Mexican (+4)', 'Matched preference: High protein (+3)', 'Matched preference: Contains chicken (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Southwestern Chicken Marinade
['Matched preference: Mexican (+4)', 'Matched preference: High protein (+3)', 'Matched preference: Contains chicken (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Garlic Ranch Chicken
['Matched preference: Mexican (+4)', 'Matched preference: High protein (+3)', 'Matched preference: Contains chicken (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Mexica

## User #2: Vegan student

Young woman, also in the 20s range, following a vegan diet. Prefferences include: trendy café food, tofu/rice bowls, smoothies, colorful vegetable-heavy meals. Has as goal to maintain mass, maybe tone.

In [62]:
vegan_student = {
    "hard_filters": {
        "diet_is_vegan": 1
    },

    "preferences": {
        'ingredient_has_tofu': 9,
        'ingredient_has_legumes': 8, 
        'ingredient_has_vegetables': 10, 
        'cuisine_is_asian': 3, 
        'duration_is_quick_meal': 4,
        'complexity_is_budget_friendly': 4,
        }
}


In [63]:
show_reccommendations(vegan_student)

Veggies and Rice Noodles Starring a Spicy Peanut Sauce
['Matched preference: Contains vegetables (+10)', 'Matched preference: Contains tofu (+9)', 'Matched preference: Contains legumes (+8)', 'Matched preference: Quick meal (+4)', 'Matched preference: Budget friendly (+4)']
------------------------------------------------------------
Quinoa Tofu Salad
['Matched preference: Contains vegetables (+10)', 'Matched preference: Contains tofu (+9)', 'Matched preference: Contains legumes (+8)', 'Matched preference: Quick meal (+4)', 'Matched preference: Budget friendly (+4)']
------------------------------------------------------------
Wilted Asian Greens
['Matched preference: Contains vegetables (+10)', 'Matched preference: Contains tofu (+9)', 'Matched preference: Contains legumes (+8)', 'Matched preference: Quick meal (+4)', 'Matched preference: Budget friendly (+4)']
------------------------------------------------------------
Panko Tofu
['Matched preference: Contains vegetables (+10)', 'Ma

## User #3: Middle aged, on low sodium diet

In [64]:
low_sodium_user = {
    "hard_filters": {
        "diet_is_low_sodium": 1
    },

    "preferences": {
        "meal_type_is_complete_meal": 3,
        'diet_is_low_calorie': 6,
        'diet_is_low_fat': 5 , 
        'ingredient_has_chicken': 3,
        'diet_is_healthy': 9, 
        'meal_type_is_soup': 2, 
        'meal_type_is_salad': 4, 
        'ingredient_has_vegetables': 5,
        'cuisine_is_mediterranean': 4, 
        'cuisine_is_greek': 3,
        #dislikes
        'cooking_is_fried': -8
    }
}


In [65]:
show_reccommendations(low_sodium_user)

Naxos Island Salad
['Matched preference: Healthy (+9)', 'Matched preference: Low calorie (+6)', 'Matched preference: Low fat (+5)', 'Matched preference: Contains vegetables (+5)', 'Matched preference: Salad (+4)']
------------------------------------------------------------
Greek Crock Pot Chicken Thighs
['Matched preference: Healthy (+9)', 'Matched preference: Low calorie (+6)', 'Matched preference: Low fat (+5)', 'Matched preference: Contains vegetables (+5)', 'Matched preference: Mediterranean (+4)']
------------------------------------------------------------
Crock Pot Greek Chicken and Potatoes
['Matched preference: Healthy (+9)', 'Matched preference: Low calorie (+6)', 'Matched preference: Low fat (+5)', 'Matched preference: Contains vegetables (+5)', 'Matched preference: Mediterranean (+4)']
------------------------------------------------------------
Bean Soup (Fasolatha)
['Matched preference: Healthy (+9)', 'Matched preference: Low calorie (+6)', 'Matched preference: Low fat (

## User #4: Gluten Free, wanting to put on muscle mass

In [66]:
gf_muscle_mass = {
    "hard_filters": {
        "diet_is_gluten_free": 1
    },

    "preferences": {
        "diet_is_high_protein": 3,
        "ingredient_has_rice": 2,
        "ingredient_has_potatoes": 2,
        "meal_type_is_complete_meal": 2,
        "meal_type_is_dessert": -3,
    }
}


In [67]:
show_reccommendations(gf_muscle_mass)

Chicken Curry
['Matched preference: High protein (+3)', 'Matched preference: Contains rice (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Onion Rice With Peas and Fried Potatoes
['Matched preference: Contains rice (+2)', 'Matched preference: Contains potatoes (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Karmir Pilaf
['Matched preference: Contains rice (+2)', 'Matched preference: Contains potatoes (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Gluten-Free Japanese Curry
['Matched preference: Contains rice (+2)', 'Matched preference: Contains potatoes (+2)', 'Matched preference: Complete meal (+2)']
------------------------------------------------------------
Middle Eastern Chickpea & Rice Stew
['Matched preference: Contains rice (+2)', 'Matched preference: Contains potatoes (+2)', 'Matched preferenc

## User #5: Gourmand person, wants to lose weight

In [68]:
gourmand_weight_loss = {
    "hard_filters": {
        "diet_is_healthy": 1
    },

    "preferences": {
        "diet_is_low_calorie": 3,
        "taste_is_savory": 3,
        "cuisine_is_french": 2,
        "cuisine_is_italian": 2,
        "ingredient_has_mushrooms": 2,
        "ingredient_has_cheese": 1,
        "meal_type_is_dessert": -2,
    }
}

In [69]:
show_reccommendations(gourmand_weight_loss)

Baked Eggplant With Mushroom-And-Tomato Sauce
['Matched preference: Low calorie (+3)', 'Matched preference: Savory (+3)', 'Matched preference: Italian (+2)', 'Matched preference: Contains mushrooms (+2)', 'Matched preference: Contains cheese (+1)']
------------------------------------------------------------
Tofu and Mushroom Marsala
['Matched preference: Low calorie (+3)', 'Matched preference: Savory (+3)', 'Matched preference: Italian (+2)', 'Matched preference: Contains mushrooms (+2)']
------------------------------------------------------------
Fettuccine Carbonara
['Matched preference: Low calorie (+3)', 'Matched preference: Savory (+3)', 'Matched preference: Italian (+2)', 'Matched preference: Contains mushrooms (+2)']
------------------------------------------------------------
Portabella Mushroom Pizza
['Matched preference: Low calorie (+3)', 'Matched preference: Savory (+3)', 'Matched preference: Italian (+2)', 'Matched preference: Contains mushrooms (+2)']
------------------